## 1. Session Overview

This notebook fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` using the MaxText framework on Kaggle TPU v5e.

- Objective: Run a minimal verification on TPU v5e, then proceed to fine-tuning.
- Evidence of Done: Successful `steps: 1` MaxText run and logs confirming TPU utilization.
- Artifacts: Config file, logs, and checkpoints saved to Kaggle outputs and/or GCS.

Preconditions:
- Kaggle accelerator set to TPU v5e.
- Internet enabled for cloning and dependency installs.
- Access to MaxText-compatible Llama 3.1 checkpoint via Kaggle Datasets.


## 2. Kaggle TPU v5e Environment Setup Plan

Steps in this session:
1. Verify TPU visibility and JAX version
2. Clone MaxText (main branch)
3. Install dependencies from `requirements.txt`
4. Prepare minimal `config.yaml` for verification run
5. Run a 1-step verification to confirm TPU v5e works

Notes:
- No substeps for now; each step maps to a single cell or small group of cells.
- We will capture logs and versions for reproducibility.


In [1]:
# 3. Verify TPU visibility and JAX environment
import os, sys, platform, subprocess, json

print("Python:", sys.version)
print("Platform:", platform.platform())

# Kaggle TPU env vars
for key in ["TPU_NAME", "TPU_WORKER_ID", "TPU_CHIPS_PER_PROCESS", "TPU_MULTISLICE_CTRL_ADDRESS"]:
    if key in os.environ:
        print(f"{key}:", os.environ[key])

try:
    import jax
    import jaxlib
    import jax.numpy as jnp
    print("jax:", jax.__version__)
    print("jaxlib:", jaxlib.__version__)
    devices = jax.devices()
    print("Devices:")
    for d in devices:
        print(" -", d)
    print("Device count:", len(devices))
    x = jnp.ones((8, 8))
    y = jnp.dot(x, x).block_until_ready()
    print("JAX test dot result shape:", y.shape)
except Exception as e:
    print("[ERROR] JAX/TPU verification failed:", e)
    raise


Python: 3.10.18 (main, Jul  1 2025, 05:26:40) [GCC 12.2.0]
Platform: Linux-6.1.42+-x86_64-with-glibc2.36
TPU_WORKER_ID: 0
jax: 0.4.34
jaxlib: 0.4.34


E0000 00:00:1758030185.949416      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


Devices:
 - TPU_0(process=0,(0,0,0,0))
 - TPU_1(process=0,(1,0,0,0))
 - TPU_2(process=0,(0,1,0,0))
 - TPU_3(process=0,(1,1,0,0))
 - TPU_4(process=0,(0,2,0,0))
 - TPU_5(process=0,(1,2,0,0))
 - TPU_6(process=0,(0,3,0,0))
 - TPU_7(process=0,(1,3,0,0))
Device count: 8
JAX test dot result shape: (8, 8)


## 4. Clone MaxText (main branch)

We will clone the official `google/maxtext` repository at the default `main` branch for the latest TPU v5e-compatible training scripts. Evidence of done: repository present in the working directory and HEAD commit printed.


In [2]:
%%bash
set -e

echo "Cloning google/maxtext (main)..."
if [ ! -d "maxtext" ]; then
  git clone --depth=1 https://github.com/google/maxtext.git
else
  echo "Repository 'maxtext' already exists; skipping clone."
fi

cd maxtext
echo "Repo HEAD:"
git log -1 --pretty=oneline || true

echo "Top-level files:"
ls -1 | sed -n '1,50p'


Cloning google/maxtext (main)...


Cloning into 'maxtext'...


Repo HEAD:
a55e18af31a76179e589314878af0a5195e7d7bd Merge pull request #2278 from AI-Hypercomputer:collabs-examples-sft
Top-level files:
AUTHORS
CONTRIBUTING.md
LICENSE
PREFLIGHT.md
README.md
RESTRUCTURE.md
benchmarks
clean_py_env.Dockerfile
code_style.sh
docker_build_dependency_image.sh
docker_upload_runner.sh
docs
download_dataset.sh
end_to_end
gpu_multi_process_run.sh
maxtext_custom_wheels.Dockerfile
maxtext_db_dependencies.Dockerfile
maxtext_dependencies.Dockerfile
maxtext_gpu_dependencies.Dockerfile
maxtext_jax_ai_image.Dockerfile
maxtext_libtpu_path.Dockerfile
maxtext_runner.Dockerfile
multihost_job.py
multihost_runner.py
pedagogical_examples
preflight.sh
pylintrc
pyproject.toml
pytest.ini
requirements.txt
requirements_docs.txt
requirements_with_jax_ai_image.txt
requirements_with_jax_stable_stack_0_6_1_pipreqs.txt
rto_setup.sh
setup.sh
setup_gcsfuse.sh
setup_with_retries.sh
src
tests
unit_test_and_lint.sh


## 7. Notes: Handling TensorFlow conflicts on TPU v5e

- Import order: Avoid importing TensorFlow before JAX; it can block TPU init.
- If TF causes conflicts but is not needed for JAX training, consider uninstalling `tensorflow` and using `tensorflow-cpu` instead.
- Keep JAX/jaxlib versions consistent with preinstalled TPU runtime.
- Evidence to capture on failure: exact import stack, package versions, and full error logs.


## 5. Install dependencies from requirements.txt

Install Python dependencies required by MaxText. Kaggle TPU v5e includes a modern JAX stack; if a conflict arises, we will prefer the preinstalled JAX. Evidence of done: successful pip install and import checks.


In [3]:
%%bash
set -e

echo "Updating apt and installing pkg-config..."
apt-get update && apt-get install -y pkg-config

echo "Upgrading pip..."
pip install --upgrade pip

echo "Installing MaxText requirements..."
# Now run the pip install command, which should find the newly installed pkg-config
pip install --no-input --no-cache-dir -r maxtext/requirements.txt

echo "Verifying JAX installation post-install..."
python - <<'PY'
import jax, jaxlib
print("jax:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("JAX import successful after requirements install.")
PY

Updating apt and installing pkg-config...
Get:1 http://deb.debian.org/debian bookworm InRelease [151 kB]
Get:2 http://deb.debian.org/debian bookworm-updates InRelease [55.4 kB]
Get:3 http://deb.debian.org/debian-security bookworm-security InRelease [48.0 kB]
Get:4 http://deb.debian.org/debian bookworm/main amd64 Packages [8791 kB]
Get:5 http://deb.debian.org/debian bookworm-updates/main amd64 Packages.diff/Index [21.8 kB]
Ign:5 http://deb.debian.org/debian bookworm-updates/main amd64 Packages.diff/Index
Get:6 http://deb.debian.org/debian-security bookworm-security/main amd64 Packages [278 kB]
Get:7 http://deb.debian.org/debian bookworm-updates/main amd64 Packages [6924 B]
Fetched 9353 kB in 1s (9522 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
pkg-config is already the newest version (1.8.1-1).
pkg-config set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 103 not upgraded.
Upgrading pip...
     ━

Installing MaxText requirements...
     \ 538.6 kB 4.2 MB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     \ 2.8 MB 6.5 MB/s 0:00:000m
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     - 417.8 kB 11.6 MB/s 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     \ 2.1 MB 13.3 MB/s 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml

  DEPRECATION: Building 'google-jetstream' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'google-jetstream'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for google-jetstream: filename=google_jetstream-0.3.0-py3-none-any.whl size=166546 sha256=ae859d3452ccbc93191d097b527fe5e3c07e8f1e21d8982e6aae4743c3365e50
  Stored in directory: /tmp/pip-ephem-wheel-cache-fjl0qogg/wheels/6d/a5/8c/89bbffd4a79016ac6755e4f78e9ad4872de0f001a7fdc17468


  DEPRECATION: Building 'mlperf-logging' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'mlperf-logging'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for mlperf-logging: filename=mlperf_logging-4.1.27-py3-none-any.whl size=321050 sha256=ce48f6748f2a9764f5ed27c3a61a4d8f72f372f64a1e469f7d5a8fce0c727530
  Stored in directory: /tmp/pip-ephem-wheel-cache-fjl0qogg/wheels/ba/e6/5e/97c034cf6620d251bb2d641152b8e0e88ba0a43d626124fd48
  Created wheel for datasets: filename=datasets-4.0.1.dev0-py3-none-any.whl size=495269 sha256=46b61094041102c445205e8209d3dc40336f26d4a0707dc3741849e8d4961ae2
  Stored in directory: /tmp/pip-ephem-wheel-cache-fjl0qogg/wheels/06/51/a1/d81445d55268567f27f77f13b405af5083e61753be42c785bc


  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334


  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144591 sha256=8a67009247fbd40ff53c2ccca1833d878c3ac4b291b4c9aab59af3b6d61a0172
  Stored in directory: /tmp/pip-ephem-wheel-cache-fjl0qogg/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built tunix google-jetstream mlperf-logging datasets antlr4-python3-runtime
  Attempting uninstall: keras
    Found existing installation: keras 3.10.0
    Uninstalling keras-3.10.0:
      Successfully uninstalled keras-3.10.0━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]
  Attempting uninstall: flatbuffers━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]━━━━━━━━━━━   1/118 [keras]
    Found existing installation: flatbuffers 25.2.102m  1/118 [keras]
    Uninstalling flatbuffer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-hub 0.21.1 requires keras>=3.5, but you have keras 2.9.0 which is incompatible.
tensorflow-tpu 2.18.0 requires flatbuffers>=24.3.25, but you have flatbuffers 1.12 which is incompatible.
tensorflow-tpu 2.18.0 requires keras>=3.5.0, but you have keras 2.9.0 which is incompatible.
tensorflow-tpu 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-tpu 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.32.1 which is incompatible.
tensorflow-tpu 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.9.0 which is incompatible.


Verifying JAX installation post-install...
jax: 0.4.34
jaxlib: 0.4.34
JAX import successful after requirements install.


## 6. Run 1-step MaxText verification

We will run a single training step using synthetic data. This should initialize JAX on TPU v5e and produce minimal logs without requiring a dataset.

Evidence of done:
- Process completes without error
- Logs show TPU devices used
- Step 1/1 completes


In [ ]:
# Generate minimal config inline and save to /kaggle/working
from pathlib import Path

config_text = """
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
dataset_type: "synthetic"
steps: 1
per_device_batch_size: 1
""".strip() + "\n"

out_path = Path("/kaggle/working/verification_minimal.yml")
out_path.write_text(config_text)
print("Wrote config to:", out_path)
print("\n--- Config ---\n" + out_path.read_text())


In [ ]:
# Run 1-step verification
import os, subprocess, sys

# Prefer running from the cloned repo if present
repo_dir = "maxtext"
train_entry_candidates = [
    os.path.join(repo_dir, "multihost_runner.py"),
    os.path.join(repo_dir, "multihost_job.py"),
    os.path.join(repo_dir, "src", "maxtext", "train.py"),
]

config_path = "/kaggle/working/verification_minimal.yml"

entry = None
for c in train_entry_candidates:
    if os.path.exists(c):
        entry = c
        break

if entry is None:
    raise FileNotFoundError("Could not find a MaxText training entrypoint.")

print("Using entrypoint:", entry)
cmd = [sys.executable, entry, config_path]
print("Running:", " ".join(cmd))

proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print("[ERROR] Verification run failed with return code", proc.returncode)
    raise SystemExit(proc.returncode)
else:
    print("[OK] Verification run completed.")
